<a href="https://colab.research.google.com/github/hminh293/hminh293/blob/main/CS338_Q22_23520955_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
# Create plots directory if it doesn't exist
os.makedirs('plots', exist_ok=True)

## Ex1. Encoding-LIF-STDP functions

###1.1 Encoding: Rate Coding

In [ ]:
def poisson_encode(signal, dt=1e-3):
    """
    Converts a continuous signal into a Poisson spike train.
    The signal represents the firing rate in Hz.
    """
    # Probability of a spike in each time step dt
    prob = signal * dt
    # Generate random numbers and compare with probability
    spike_train = (np.random.rand(*signal.shape) < prob).astype(float)
    return spike_train

###  1.2 Leaky Integrate-and-Fire (LIF) Neuron Model

In [ ]:
class LIFNeuron:
    def __init__(self, tau_m=20.0, v_rest=0.0, v_reset=0.0, v_threshold=1.0, R=1.0, dt=1.0):
        self.tau_m = tau_m        # Membrane time constant (ms)
        self.v_rest = v_rest      # Resting potential (mV)
        self.v_reset = v_reset    # Reset potential (mV)
        self.v_threshold = v_threshold # Spiking threshold (mV)
        self.R = R                # Resistance (Ohm-ish)
        self.dt = dt              # Time step (ms)
        self.v = v_rest           # Current membrane potential

    def update(self, I_ext):
        """
        Updates the membrane potential based on input current.
        Returns 1 if a spike occurred, 0 otherwise.
        """
        # Discrete LIF equation: dv/dt = (-(v - v_rest) + R*I) / tau_m
        dv = (-(self.v - self.v_rest) + self.R * I_ext) / self.tau_m * self.dt
        self.v += dv

        spike = 0
        if self.v >= self.v_threshold:
            spike = 1
            self.v = self.v_reset # Reset after spike

        return spike, self.v

### 1.3 Spike-Timing-Dependent Plasticity (STDP)

In [ ]:
def stdp_update(delta_t, tau_plus=20.0, tau_minus=20.0, A_plus=0.1, A_minus=0.12):
    """
    Calculates the weight change delta_w based on the time difference
    between pre-synaptic and post-synaptic spikes.
    delta_t = t_post - t_pre
    """
    if delta_t > 0:
        # LTP (Long-Term Potentiation): pre before post
        return A_plus * np.exp(-delta_t / tau_plus)
    elif delta_t < 0:
        # LTD (Long-Term Depression): post before pre
        return -A_minus * np.exp(delta_t / tau_minus)
    else:
        return 0.0

### Simulation and Visualization for Ex1

####  1.1 Test Encoding

In [ ]:
t = np.arange(0, 1000, 1) # 1000 ms
# Sine wave signal (scale to Hz)
signal = 50 * (np.sin(2 * np.pi * t / 500) + 1)
spikes = poisson_encode(signal)

plt.figure(figsize=(10, 4))
plt.subplot(2, 1, 1)
plt.plot(t, signal)
plt.title("1.1 Continuous Signal (Sine Wave)")
plt.ylabel("Rate (Hz)")
plt.subplot(2, 1, 2)
plt.eventplot(np.where(spikes > 0)[0], colors='black')
plt.title("1.1 Poisson Spike Train")
plt.xlabel("Time (ms)")
plt.tight_layout()
plt.savefig('plots/ex1_1_encoding.png')
plt.close()

#### 1.2 Test LIF Neuron

In [ ]:
neuron = LIFNeuron(tau_m=20.0, v_threshold=1.0, R=1.0)
v_trace = []
spike_times = []
input_current = 1.5 # Constant step current
for ts in t:
    spike, v = neuron.update(input_current)
    v_trace.append(v)
    if spike:
        spike_times.append(ts)

plt.figure(figsize=(10, 4))
plt.plot(t, v_trace, label="Membrane Potential")
plt.axhline(y=1.0, color='r', linestyle='--', label="Threshold")
plt.vlines(spike_times, ymin=0, ymax=1.2, colors='g', alpha=0.5, label="Spikes")
plt.title("1.2 LIF Neuron Response to Step Current (I=1.5)")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.legend()
plt.tight_layout()
plt.savefig('plots/ex1_2_lif.png')
plt.close()


#### 1.3 Test STDP

In [ ]:
delta_ts = np.linspace(-100, 100, 400)
dw = [stdp_update(dt) for dt in delta_ts]

plt.figure(figsize=(10, 4))
plt.plot(delta_ts, dw, color='blue')
plt.axhline(0, color='black', lw=1)
plt.axvline(0, color='black', lw=1)
plt.title("1.3 STDP Weight Change Curve")
plt.xlabel("t_post - t_pre (ms)")
plt.ylabel("Weight Change (\u0394w)")
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/ex1_3_stdp.png')
plt.close()

print("Exercise 1 simulations complete. Plots saved in 'plots/'.")

Exercise 1 simulations complete. Plots saved in 'plots/'.


# EX2.Implement Spiking CNN for Image Recognition (CSNN) on MNIST (handwritten digits).

## 2.1 Rate-Based Encoding (NumPy Version)

In [ ]:
def rate_encode(image, num_steps=30, max_rate=0.8):
    """
    image: (28, 28) normalized to [0, 1]
    returns: (num_steps, 28, 28) spike train
    """
    # Probability of spike at each pixel
    probs = image * max_rate
    spikes = (np.random.rand(num_steps, *image.shape) < probs).astype(float)
    return spikes

##2.2 LIF Neuron for Layers

In [ ]:
class LIFNeuronLayer:
    def __init__(self, shape, tau_m=20.0, v_threshold=1.0, v_reset=0.0):
        self.v = np.zeros(shape)
        self.tau_m = tau_m
        self.v_threshold = v_threshold
        self.v_reset = v_reset

    def update(self, I_ext):
        # Leaky integrate
        self.v += (-self.v / self.tau_m) + I_ext
        # Fire
        spikes = (self.v >= self.v_threshold).astype(float)
        # Reset
        self.v[spikes > 0] = self.v_reset
        return spikes

## 2.3 Manual Convolutional Layer (Simplified)

In [ ]:
def conv2d(input_spikes, kernel):
    """
    input_spikes: (H, W)
    kernel: (KH, KW)
    """
    h, w = input_spikes.shape
    kh, kw = kernel.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(input_spikes[i:i+kh, j:j+kw] * kernel)

    return output

# ---------------------------------------------------------
# 2.4 Simulation of CSNN Inference
# ---------------------------------------------------------
# Simulate an MNIST image (Digit 7)
digit_7 = np.zeros((28, 28))
digit_7[5:10, 5:23] = 1.0 # Top bar
digit_7[10:25, 18:22] = 1.0 # Stem

# Parameters
num_steps = 50
# Random kernel for convolution (acting as a feature detector)
kernel = np.random.randn(5, 5) * 0.1

# 1. Encode image to spike train
spike_train = rate_encode(digit_7, num_steps=num_steps)

# 2. Setup Layers
lif_layer = LIFNeuronLayer((24, 24)) # After 5x5 conv, 28x28 becomes 24x24
output_spikes_total = np.zeros((24, 24))

# 3. Process over time
for t in range(num_steps):
    # Synaptic input from convolution
    synaptic_input = conv2d(spike_train[t], kernel)
    # Update LIF neurons
    layer_spikes = lif_layer.update(synaptic_input)
    # Accumulate spikes
    output_spikes_total += layer_spikes

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 3, 1)
plt.imshow(digit_7, cmap='gray')
plt.title("Original Image (Digit 7)")

plt.subplot(1, 3, 2)
# Show mean firing rate in the hidden layer
plt.imshow(output_spikes_total / num_steps, cmap='hot')
plt.title("SNN Feature Map (Firing Rate)")

plt.subplot(1, 3, 3)
# Show spikes at a single time step
plt.imshow(spike_train[10], cmap='gray')
plt.title("Input Spikes at t=10")

plt.tight_layout()
plt.savefig('plots/ex2_scnn_mnist_numpy.png')
plt.close()

print("Exercise 2 (NumPy version) simulation complete.")

Exercise 2 (NumPy version) simulation complete.
